## Evaluación Parcial N°2

Asignatura: Preprocesamiento de Datos

Integrantes: 
* Alejandra González
* Constanza González
* Diego Villar

Sección: 800D

In [1]:
import sys
!{sys.executable} -m pip install matplotlib seaborn


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## [Celda 1] Carga de librerías y lectura del dataset

En esta etapa, se realiza la ingesta del dataset original. Es fundamental especificar el separador adecuado (; en lugar de la coma estándar) para asegurar que el motor de Pandas estructure correctamente el DataFrame. Se realiza una inspección inicial de las dimensiones para validar la integridad del archivo importado.

In [10]:
# Importación de librerías base para la manipulación de datos
import pandas as pd
import numpy as np

# Carga del dataset original
# Se especifica el separador ';' ya que el archivo no usa comas estándar
df = pd.read_csv('bank-additional-full.csv', sep=';')

# Verificamos la forma inicial del dataset y las primeras filas
print(f"Dimensiones iniciales: {df.shape}")
df

Dimensiones iniciales: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


## [Celda 2] Estandarización de formatos y traducción

Para cumplir con los requerimientos de la organización y garantizar una comunicación efectiva con el cliente, se estandarizan las cadenas de texto (eliminación de caracteres especiales en categorías como admin.) y se traducen las variables temporales (month y day_of_week) al español. Esto asegura que las visualizaciones gráficas posteriores sean claras, profesionales y fáciles de interpretar para los stakeholders del banco.

In [3]:
# Limpieza de strings: quitamos el punto en la categoría 'admin.' para estandarizar
df['job'] = df['job'].str.replace('admin.', 'admin', regex=False)

# Diccionarios de traducción para mejorar la presentación al cliente en los gráficos
meses_es = {
    'may': 'mayo', 'jun': 'junio', 'jul': 'julio', 'aug': 'agosto', 
    'oct': 'octubre', 'nov': 'noviembre', 'dec': 'diciembre', 
    'mar': 'marzo', 'apr': 'abril', 'sep': 'septiembre'
}
dias_es = {
    'mon': 'lunes', 'tue': 'martes', 'wed': 'miércoles', 
    'thu': 'jueves', 'fri': 'viernes'
}

# Aplicamos la traducción a las columnas correspondientes
df['month'] = df['month'].map(meses_es)
df['day_of_week'] = df['day_of_week'].map(dias_es)

print("Traducción y estandarización completada.")

Traducción y estandarización completada.


## [Celda 3] Gestión Avanzada de Datos Faltantes (Valores "Unknown")

El dataset presenta valores desconocidos que reflejan información ausente. En lugar de una imputación global que introduciría sesgos, aplicamos una estrategia mixta basada en lógica de negocio:  

Imputación Condicional: La educación y el trabajo están correlacionados. Imputamos los valores "unknown" de education basándonos en la moda específica del grupo de job al que pertenece el cliente.

Retención de Señal de Riesgo: Para variables financieras como default (mora), el valor "unknown" se mantiene, ya que en el sector bancario la omisión de esta información es un indicador de riesgo crediticio que el modelo debe aprender.

In [4]:
# Imputación condicional: Llenar 'unknown' en education según la moda del 'job'
def imputar_educacion(row):
    if row['education'] == 'unknown':
        moda_grupo = df[(df['job'] == row['job']) & (df['education'] != 'unknown')]['education'].mode()
        if not moda_grupo.empty:
            return moda_grupo[0]
    return row['education']

df['education'] = df.apply(imputar_educacion, axis=1)

print("Imputación condicional en educación aplicada.")
print("Variables financieras y estado civil conservan sus valores originales para evitar sesgos analíticos.")

Imputación condicional en educación aplicada.
Variables financieras y estado civil conservan sus valores originales para evitar sesgos analíticos.


## [Celda 4] Reducción de cardinalidad

Para optimizar el rendimiento de los algoritmos y mejorar la claridad visual de los gráficos, agrupamos categorías fragmentadas. Específicamente, los distintos niveles de educación básica (basic.4y, basic.6y, basic.9y) y analfabetismo se consolidan en una única categoría macro (basic).

In [5]:
# Agrupamos 'basic.4y', 'basic.6y', 'basic.9y' y la micro-categoría 'illiterate'
# en una sola categoría macro: 'basic'
categorias_basicas = ['basic.4y', 'basic.6y', 'basic.9y', 'illiterate']
df['education'] = df['education'].replace(categorias_basicas, 'basic')

print("Nuevas categorías en educación:")
print(df['education'].unique())

Nuevas categorías en educación:
<StringArray>
['basic', 'high.school', 'professional.course', 'university.degree']
Length: 4, dtype: str


## [Celda 5] Transformaciones, Discretización y Ética

Se realizan transformaciones clave para el análisis de negocio:

Pdays: Se transforma el outlier estructural 999 (cliente no contactado) en una variable binaria de negocio (contactado_previamente), aportando una dimensión clara de fidelización.  

Discretización (Binning): Se agrupa la variable age (Edad) en segmentos generacionales para facilitar el descubrimiento de patrones en el análisis exploratorio.

Ética y Viabilidad: La variable duration afecta directamente el objetivo y no es conocida a priori. Se conservará temporalmente para analizar eficiencias operativas en el EDA, pero debe ser eliminada de los conjuntos de entrenamiento de Machine Learning para evitar fuga de información (data leakage).

In [6]:
# 1. Transformación de pdays
df['contactado_previamente'] = np.where(df['pdays'] == 999, 0, 1)
df['pdays'] = df['pdays'].replace(999, np.nan)

# 2. Nueva característica: Segmentación de edad (Binning)
bins = [16, 30, 45, 60, 100]
labels = ['Jóvenes (<30)', 'Adultos (30-45)', 'Maduros (45-60)', 'Tercera Edad (60+)']
df['rango_edad'] = pd.cut(df['age'], bins=bins, labels=labels)

# 3. Mantenemos 'duration' solo para los gráficos, pero dejamos constancia.
print("Nuevas variables creadas. NOTA: 'duration' debe eliminarse antes del modelado.")

Nuevas variables creadas. NOTA: 'duration' debe eliminarse antes del modelado.


## [Celda 6] Tratamiento de Outliers (Límite en llamadas por campaña)

Evaluamos variables operativas como campaign (número de contactos). Aplicamos un límite superior usando el Rango Intercuartílico (IQR) para mitigar el efecto de valores extremos (clientes sobre-contactados) sin incurrir en la pérdida de registros valiosos. Los índices macroeconómicos no se limitan, ya que representan la realidad económica del momento.

In [7]:
# La variable 'campaign' tiene clientes con un número extremo de contactos.
# Usaremos el Rango Intercuartílico (IQR) para limitar (capping) estos valores extremos 
# y evitar sesgos sin eliminar filas.

Q1 = df['campaign'].quantile(0.25)
Q3 = df['campaign'].quantile(0.75)
IQR = Q3 - Q1

# Definimos el límite superior
limite_superior = Q3 + 1.5 * IQR

# Aplicamos capping: todo valor mayor al límite superior se iguala al límite superior
df['campaign'] = np.where(df['campaign'] > limite_superior, limite_superior, df['campaign'])

print(f"Outliers en 'campaign' limitados a un máximo de {limite_superior} contactos.")

Outliers en 'campaign' limitados a un máximo de 6.0 contactos.


## [Celda 7] Codificación numérica y Exportación

Se binariza la variable objetivo y (yes = 1, no = 0) para posibilitar el cálculo de correlaciones y promedios en las métricas de conversión. Finalmente, se exporta el DataFrame procesado, garantizando que el análisis visual posterior se realice sobre datos consistentes e íntegros.

In [8]:
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# 1. Exportamos el dataset los gráficos (Mantiene la columna 'duration' para análisis operativo)
df.to_csv('bank_limpio_para_graficos.csv', index=False)
print("Archivo 'bank_limpio_para_graficos.csv' exportado.")

# 2. Exportamos el dataset para Machine Learning (Ético y sin Data Leakage)
df_ml = df.drop(columns=['duration'])
df_ml.to_csv('bank_limpio_para_ml.csv', index=False)
print("Archivo 'bank_limpio_para_ml.csv' exportado SIN 'duration'. Listo y seguro para algoritmos.")

Archivo 'bank_limpio_para_graficos.csv' exportado.
Archivo 'bank_limpio_para_ml.csv' exportado SIN 'duration'. Listo y seguro para algoritmos.


## [Celda 8] Traducción Final para Presentación Visual

Para cumplir con los estándares de calidad en la presentación al cliente y garantizar una comunicación clara, traducimos las variables categóricas restantes del inglés al español. Esto asegura que todas las leyendas y ejes en las visualizaciones de Matplotlib y Seaborn sean inmediatamente comprensibles para los tomadores de decisiones del banco.

In [9]:
# Diccionarios de traducción para variables binarias y de estado
trad_sino = {'yes': 'sí', 'no': 'no', 'unknown': 'desconocido'}
trad_contacto = {'telephone': 'teléfono fijo', 'cellular': 'celular'}
trad_resultado = {'nonexistent': 'inexistente', 'failure': 'fracaso', 'success': 'éxito'}

# Aplicamos las traducciones a las columnas correspondientes
df['housing'] = df['housing'].map(trad_sino).fillna(df['housing'])
df['loan'] = df['loan'].map(trad_sino).fillna(df['loan'])
df['default'] = df['default'].map(trad_sino).fillna(df['default'])
df['contact'] = df['contact'].map(trad_contacto).fillna(df['contact'])
df['poutcome'] = df['poutcome'].map(trad_resultado).fillna(df['poutcome'])

# Actualizamos el archivo final
df.to_csv('bank_limpio_para_graficos.csv', index=False)
print("Traducciones visuales aplicadas. Dataset 100% listo para graficar.")

Traducciones visuales aplicadas. Dataset 100% listo para graficar.
